# Paper Tables

Generate LaTeX tables for the paper.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

## Helper Functions

In [2]:
def load_run(fpath):
    """Load results from a single run."""
    fpath = Path(fpath)
    if not fpath.exists():
        return None
    with open(fpath) as f:
        return [json.loads(l) for l in f]

def compute_unified_metrics(entries):
    """Compute metrics for unified approach (baseline, directed, nudged)."""
    total = len(entries)
    lean_pass = sum(1 for e in entries if e.get("success") or e.get("lean_verification", {}).get("success", False))
    correct = sum(1 for e in entries if e.get("correct"))
    return {
        "compilation": lean_pass / total * 100 if total > 0 else 0,
        "accuracy": correct / total * 100 if total > 0 else 0
    }

def compute_twostage_metrics(entries):
    """Compute metrics for two-stage approach."""
    total = len(entries)
    s1_pass = sum(1 for e in entries if e.get("stage1_success"))
    s2_pass = sum(1 for e in entries if e.get("stage2_success"))
    correct = sum(1 for e in entries if e.get("correct"))
    return {
        "stage1": s1_pass / total * 100 if total > 0 else 0,
        "stage2": s2_pass / total * 100 if total > 0 else 0,
        "accuracy": correct / total * 100 if total > 0 else 0
    }

def fmt(values):
    """Format mean±std for LaTeX."""
    mean = np.mean(values)
    std = np.std(values)
    return f"{mean:.1f}\\tiny{{±{std:.1f}}}"

# Configuration
CONDITIONS = [
    ("Baseline", "baseline"),
    ("Directed$_\\text{T}$", "bidir_true"),
    ("Directed$_\\text{F}$", "bidir_false"),
    ("Nudged$_\\text{T}$", "spooky_true"),
    ("Nudged$_\\text{F}$", "spooky_false"),
]

MODELS = [("GPT-5", "gpt-5"), ("DeepSeek-R1", "deepseek")]
DATASETS = [("FOLIO", "folio"), ("M-LogiEval", "multilogieval")]

## Table 1: FOLIO Results (mean±std over 3 runs)

In [3]:
# FOLIO Results
print("FOLIO RESULTS (mean±std over 3 runs)")
print("=" * 70)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Accuracy %':>15}")
print("-" * 70)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        folio_comp, folio_acc = [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/folio/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                folio_comp.append(m["compilation"])
                folio_acc.append(m["accuracy"])
        
        fc = f"{np.mean(folio_comp):.1f}±{np.std(folio_comp):.1f}"
        fa = f"{np.mean(folio_acc):.1f}±{np.std(folio_acc):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {fc:>15} {fa:>15}")
    
    # Two-Stage
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    folio_s1, folio_s2, folio_acc = [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/folio/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            folio_s1.append(m["stage1"])
            folio_s2.append(m["stage2"])
            folio_acc.append(m["accuracy"])
    
    # Show std for both stages
    fs1 = f"{np.mean(folio_s1):.1f}±{np.std(folio_s1):.1f}"
    fs2 = f"{np.mean(folio_s2):.1f}±{np.std(folio_s2):.1f}"
    fa = f"{np.mean(folio_acc):.1f}±{np.std(folio_acc):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{fs1} S2:{fs2}  {fa}")
    print()

FOLIO RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %      Accuracy %
----------------------------------------------------------------------
GPT-5          Baseline                    98.2±0.8        85.2±0.4
GPT-5          Directed$_\text{T}$         99.7±0.5        91.3±0.2
GPT-5          Directed$_\text{F}$         99.2±0.5        92.0±0.5
GPT-5          Nudged$_\text{T}$           99.5±0.4        92.1±0.4
GPT-5          Nudged$_\text{F}$           98.9±0.6        92.3±0.2
GPT-5          Two-Stage            S1:100.0±0.0 S2:81.6±1.9  72.7±4.3

DeepSeek-R1    Baseline                    94.6±1.1        86.4±0.6
DeepSeek-R1    Directed$_\text{T}$         95.1±1.1        91.0±0.5
DeepSeek-R1    Directed$_\text{F}$         92.1±1.5        91.1±0.4
DeepSeek-R1    Nudged$_\text{T}$           88.7±1.5        91.5±1.2
DeepSeek-R1    Nudged$_\text{F}$           86.5±1.3        91.3±2.0
DeepSeek-R1    Two-Stage            S1:99.2±0.6 S2:85.7±1.5  76.2±2.3



## Table 2: Multi-LogiEval Results (mean±std over 3 runs)

In [4]:
# Multi-LogiEval Results
print("MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)")
print("=" * 70)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Accuracy %':>15}")
print("-" * 70)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        multi_comp, multi_acc = [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/multilogieval/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                multi_comp.append(m["compilation"])
                multi_acc.append(m["accuracy"])
        
        mc = f"{np.mean(multi_comp):.1f}±{np.std(multi_comp):.1f}"
        ma = f"{np.mean(multi_acc):.1f}±{np.std(multi_acc):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {mc:>15} {ma:>15}")
    
    # Two-Stage
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    multi_s1, multi_s2, multi_acc = [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/multilogieval/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            multi_s1.append(m["stage1"])
            multi_s2.append(m["stage2"])
            multi_acc.append(m["accuracy"])
    
    # Show std for both stages
    ms1 = f"{np.mean(multi_s1):.1f}±{np.std(multi_s1):.1f}"
    ms2 = f"{np.mean(multi_s2):.1f}±{np.std(multi_s2):.1f}"
    ma = f"{np.mean(multi_acc):.1f}±{np.std(multi_acc):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{ms1} S2:{ms2}  {ma}")
    print()

MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %      Accuracy %
----------------------------------------------------------------------
GPT-5          Baseline                    99.3±0.9        71.7±2.1
GPT-5          Directed$_\text{T}$         99.3±0.9        85.7±1.2
GPT-5          Directed$_\text{F}$         99.3±0.5        83.7±1.2
GPT-5          Nudged$_\text{T}$           99.0±0.8        92.0±0.8
GPT-5          Nudged$_\text{F}$           99.7±0.5        85.3±1.2
GPT-5          Two-Stage            S1:100.0±0.0 S2:89.7±4.5  56.7±2.9

DeepSeek-R1    Baseline                    97.3±0.9        69.7±1.7
DeepSeek-R1    Directed$_\text{T}$         96.3±2.5        82.3±2.5
DeepSeek-R1    Directed$_\text{F}$         94.0±2.2        82.0±1.6
DeepSeek-R1    Nudged$_\text{T}$           93.3±2.4        83.0±2.4
DeepSeek-R1    Nudged$_\text{F}$           90.7±2.5        82.3±4.5
DeepSeek-R1    Two-Stage            S1:99.7±0.5 S2:95.3±0.5  64

## Structured Error Table (Compiled Cases Only)

Full breakdown of wrong predictions by Ground Truth → Prediction for each condition.

In [5]:
# Structured error table - all wrong predictions by GT -> Pred, BY DATASET and MODEL

# Simple conditions (for display purposes)
CONDITIONS_SIMPLE = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

all_wrong = []
all_compiled = []
all_total = []

# Regular conditions
for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        for cond_name, cond in CONDITIONS_SIMPLE:
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue
                for e in entries:
                    all_total.append({"dataset": dataset, "model": model, "condition": cond_name})
                    lean_success = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    if lean_success:
                        all_compiled.append({"dataset": dataset, "model": model, "condition": cond_name})
                    if not lean_success or e.get("correct", False):
                        continue
                    pred = str(e.get("prediction", "")).lower()
                    gt = str(e.get("ground_truth", "")).lower()
                    if gt in ["yes"]: gt = "true"
                    elif gt in ["no"]: gt = "false"
                    if pred in ["yes"]: pred = "true"
                    elif pred in ["no"]: pred = "false"
                    all_wrong.append({"dataset": dataset, "model": model, "condition": cond_name, "gt": gt.capitalize(), "pred": pred.capitalize()})

# Two-stage
for dataset, dataset_dir in DATASETS:
    for model_name in ["gpt-5", "deepseek-r1"]:
        model_display = "GPT-5" if model_name == "gpt-5" else "DeepSeek-R1"
        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue
            for e in entries:
                all_total.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if e.get("stage2_success", False):
                    all_compiled.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if not e.get("stage2_success", False) or e.get("correct", False):
                    continue
                pred = str(e.get("prediction", "")).lower()
                gt = str(e.get("ground_truth", "")).lower()
                if gt in ["yes"]: gt = "true"
                elif gt in ["no"]: gt = "false"
                if pred in ["yes"]: pred = "true"
                elif pred in ["no"]: pred = "false"
                all_wrong.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage", "gt": gt.capitalize(), "pred": pred.capitalize()})

df_wrong_all = pd.DataFrame(all_wrong)
df_compiled_model = pd.DataFrame(all_compiled)
df_total_model = pd.DataFrame(all_total)

error_types = ["T→F", "T→Unc", "T→Fail", "F→T", "F→Unc", "F→Fail", "Unc→T", "Unc→F"]
error_map = {
    "T→F": ("True", "False"), "T→Unc": ("True", "Uncertain"), "T→Fail": ("True", "Failure"),
    "F→T": ("False", "True"), "F→Unc": ("False", "Uncertain"), "F→Fail": ("False", "Failure"),
    "Unc→T": ("Uncertain", "True"), "Unc→F": ("Uncertain", "False")
}

print("STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)")
print("="*100)
print(f"Total wrong (compiled): {len(df_wrong_all)}")
print()

for dataset in ["FOLIO", "M-LogiEval"]:
    print(f"\n{'='*100}")
    print(f"DATASET: {dataset}")
    print(f"{'='*100}")
    
    for model_name in ["GPT-5", "DeepSeek-R1"]:
        mask_wrong = (df_wrong_all["dataset"] == dataset) & (df_wrong_all["model"] == model_name)
        mask_comp = (df_compiled_model["dataset"] == dataset) & (df_compiled_model["model"] == model_name)
        mask_total = (df_total_model["dataset"] == dataset) & (df_total_model["model"] == model_name)
        
        model_wrong = df_wrong_all[mask_wrong]
        model_compiled = df_compiled_model[mask_comp]
        model_total = df_total_model[mask_total]
        
        print(f"\n{model_name}: {len(model_wrong)} wrong / {len(model_compiled)} compiled / {len(model_total)} total")
        print("-"*100)
        
        rows = []
        for cond in ["Baseline", "Directed_T", "Directed_F", "Nudged_T", "Nudged_F", "Two-Stage"]:
            cond_df = model_wrong[model_wrong["condition"] == cond]
            cond_compiled = len(model_compiled[model_compiled["condition"] == cond])
            cond_total = len(model_total[model_total["condition"] == cond])
            row = {"Condition": cond, "Wrong/Comp/Tot": f"{len(cond_df)}/{cond_compiled}/{cond_total}"}
            for etype in error_types:
                gt_val, pred_val = error_map[etype]
                count = len(cond_df[(cond_df["gt"] == gt_val) & (cond_df["pred"] == pred_val)])
                row[etype] = count if count > 0 else "-"
            rows.append(row)
        
        error_df = pd.DataFrame(rows)
        print(error_df.to_string(index=False))

print("\n" + "="*100)
print("Legend: T=True, F=False, Unc=Uncertain, Fail=Failure (couldn't prove)")
print("Wrong/Comp/Tot = Wrong among compiled / Compiled / Total cases")

STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)
Total wrong (compiled): 1548


DATASET: FOLIO

GPT-5: 433 wrong / 3514 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     88/598/609   -    34      -   8    33      -    12     1
Directed_T     53/607/609   -     -     30   9     -      -    14     -
Directed_F     49/604/609   9     -      -   -     -     32     -     8
  Nudged_T     48/606/609   -     -     16  13     -      -    19     -
  Nudged_F     45/602/609  10     -      -   -     -     18     -    17
 Two-Stage    150/497/609  11    23      -  10    16      -    10    80

DeepSeek-R1: 383 wrong / 3305 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     77/57